# Import

In [1]:
import numpy as np
import json
from scipy.sparse import load_npz,save_npz,diags,csr_matrix
import scipy.sparse as sp
import pandas as pd
import os
import requests
from io import BytesIO
from tqdm import tqdm
from scipy.sparse.linalg import eigsh
from scipy.spatial.distance import pdist, squareform
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.backends.backend_pdf import PdfPages
from pypdf import PdfReader, PdfWriter
from tempfile import NamedTemporaryFile
import networkx as nx
import pickle
import gseapy as gp
import mygene
from IPython.display import display, HTML
import re
from collections import deque
from goatools.obo_parser import GODag
import math
from itertools import combinations
from collections import Counter
from gseapy.parser import read_gmt
import time
import random
import ast

In [2]:
pd.set_option('display.width', None)      # No line-wrapping
pd.set_option('display.max_columns', None)  # Show all columns

# Prep

## Loading variables

In [3]:
DISEASE = input("Disease: ")
DISEASE_FOLDER = f"../output/{DISEASE}/"
RESULT_FOLDER = DISEASE_FOLDER + "leiden_results"
DGIDB_DIRECTORY = f"../../Gen_Hypergraph/output/DGIDB_{DISEASE}/"
MSIGDB_DIRECTORY = "../../Gen_Hypergraph/output/MSigDB_Full/"
RESULT_GRAPH = "result_graph"

with open(DISEASE_FOLDER + "gene_to_index_distinct.json", "r") as file:
    gene_to_index_distinct = json.load(file)
    
try:
    with open(DGIDB_DIRECTORY + f"gene_to_index.json", "r") as file:
        DGIDB_gene_to_index = json.load(file)
except FileNotFoundError:
    DGIDB_gene_to_index = {}
    print("File not found. Setting DGIDB_gene_to_index to be {}.")

In [4]:
## ORIGINAL
index_to_gene_distinct = {v: k for k, v in gene_to_index_distinct.items()}

In [5]:
# Loading result graph and communities
with open(f"{RESULT_FOLDER}/result_communities_selected.pkl", "rb") as f:
    communities_selected = pickle.load(f)
with open(f"{RESULT_FOLDER}/result_communities.pkl", "rb") as f:
    communities = pickle.load(f)
with open(f"{RESULT_FOLDER}/{RESULT_GRAPH}.pkl", "rb") as f:
    graph = pickle.load(f)

In [6]:
for c in communities:
    print(len(c))

4046
3897
3069
2085
1376
1352
1268
1044
686
534
474
434
424
217
211
184
157
23
12
8
8
8
7
7
6
6
5
5
5
5
5
4
4
4
4
3
3
3
3
2
2
2
2
2
2
2
2
2
2
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1


## Helpful functions (big object, drop NAN)

In [7]:
# Helpful functions
def drop_nan_from_communities(communities):
    cleaned_communities = []
    total_dropped = 0

    for i, community in enumerate(communities):
        cleaned = []
        dropped = 0
        for g in community:
            if g is None or (isinstance(g, float) and math.isnan(g)):
                dropped += 1
            else:
                cleaned.append(g)
        cleaned_communities.append(cleaned)
        total_dropped += dropped
        print(f"Community {i}: dropped {dropped} NaN entries")

    print(f"\nTotal dropped across all communities: {total_dropped}")
    return cleaned_communities

def big_objects(n=10, min_mb=1):
    """
    Show the largest objects currently in memory.
    
    Parameters
    ----------
    n : int
        Number of top objects to show.
    min_mb : float
        Minimum size (in MB) to include.
    """
    import sys
    import numpy as np
    import pandas as pd
    import scipy.sparse as sp
    from IPython import get_ipython

    def get_size(obj):
        try:
            if isinstance(obj, np.ndarray):
                return obj.nbytes
            elif isinstance(obj, pd.DataFrame) or isinstance(obj, pd.Series):
                return obj.memory_usage(deep=True).sum()
            elif sp.issparse(obj):
                return (obj.data.nbytes +
                        obj.indptr.nbytes +
                        obj.indices.nbytes)
            else:
                return sys.getsizeof(obj)
        except Exception:
            return 0

    ip = get_ipython()
    if ip is None:
        ns = globals()
    else:
        ns = ip.user_ns

    items = []
    for name, val in ns.items():
        if name.startswith('_'):
            continue  # skip internals
        size = get_size(val)
        if size > min_mb * 1024 ** 2:
            items.append((name, type(val).__name__, size))

    items.sort(key=lambda x: x[2], reverse=True)

    print(f"{'Variable':30s} {'Type':25s} {'Size (MB)':>10s}")
    print("-" * 70)
    for name, t, size in items[:n]:
        print(f"{name:30s} {t:25s} {size / 1024 ** 2:10.2f}")

## Index to NCBI

In [8]:
# Convert index to ncbi
def index_to_ncbi(comms,index_to_ncbi_dict = index_to_gene_distinct):
    comms_ncbi = [list(map(index_to_ncbi_dict.get, c)) for c in comms]
    return comms_ncbi

In [9]:
communities_ncbi = index_to_ncbi(communities_selected,index_to_gene_distinct)
print(communities_ncbi)
print(len(communities_ncbi))
with open(f"{RESULT_FOLDER}/result_communities_ncbi_selected.pkl", "wb") as f:
    pickle.dump(communities_ncbi, f)

[['961', '9590', '3964', '22836', '4154', '23180', '8682', '8502', '9961', '26353', '9517', '23682', '29969', '6515', '6483', '6382', '6489', '6876', '7078', '9638', '9592', '2887', '6482', '51616', '310', '8436', '84525', '55959', '51278', '7357', '23576', '25959', '11156', '79589', '23433', '10923', '7205', '55660', '3613', '3159', '8611', '54453', '51776', '79366', '29775', '57103', '9891', '81610', '23210', '2530', '2012', '10788', '8507', '9367', '54843', '54509', '7837', '274', '4281', '55636', '10123', '2043', '1992', '4495', '54741', '5768', '8780', '6421', '152007', '10628', '4430', '9404', '3178', '6698', '9555', '91966', '8879', '467', '23194', '9747', '4212', '4072', '22936', '4121', '1736', '10299', '23580', '27042', '966', '9903', '23499', '253782', '51304', '8994', '6768', '3275', '9867', '83892', '3998', '427', '6385', '123872', '5782', '10553', '11030', '1827', '84159', '9639', '4681', '23258', '2274', '6453', '1847', '6651', '6303', '79971', '5865', '9262', '2896', '8

In [10]:
communities_ncbi_full = index_to_ncbi(communities,index_to_gene_distinct)
print(communities_ncbi_full)
print(len(communities_ncbi_full))
with open(f"{RESULT_FOLDER}/result_communities_ncbi.pkl", "wb") as f:
    pickle.dump(communities_ncbi_full, f)

[['360158', '11', '236', '19', '20', '21', '29', '30', '32', '33', '34', '35', '38', '39', '41', '43', '51', '55', '60', '70', '71', '81', '87', '88', '90', '91', '92', '93', '100', '101', '102', '103', '104', '111', '112', '113', '115', '116', '118', '119', '126', '128', '133', '135', '140', '142', '143', '146', '156', '157', '158', '160', '161', '163', '164', '166', '177', '181', '182', '187', '190', '191', '197', '199', '203', '204', '205', '208', '210', '211', '212', '219', '220', '221', '222', '223', '224', '225', '226', '229', '230', '231', '239', '240', '241', '242', '246', '247', '267', '274', '283', '284', '285', '286', '287', '288', '290', '291', '292', '301', '302', '306', '307', '310', '313', '314', '317', '324', '325', '328', '329', '330', '331', '335', '336', '337', '338', '341', '344', '345', '346', '347', '350', '351', '356', '358', '359', '360', '362', '364', '366', '368', '374', '378', '382', '387', '388', '389', '391', '392', '395', '396', '397', '399', '403', '408',

## NCBI to HGNC

In [11]:
hgnc = pd.read_csv("../../Data/hgnc_complete_set.txt", sep="\t", dtype=str)
ncbi_to_hgnc_dict = dict(
    zip(
        hgnc["entrez_id"].dropna(),
        hgnc.loc[hgnc["entrez_id"].notna(), "symbol"]
    )
)

def ncbi_to_HGNC(comms_ncbi):
    comms_HGNC = []
    for community in comms_ncbi:
        symbols = [ncbi_to_hgnc_dict.get(n) for n in community]
        comms_HGNC.append(symbols)
    return comms_HGNC

In [12]:
# # NCBI to HGNC symbol
# def ncbi_to_HGNC(comms_ncbi):
#     comms_HGNC = []
#     for community in comms_ncbi:
#         mg = mygene.MyGeneInfo()
#         entrez_ids = [str(e) for e in community]

#         results = mg.querymany(
#             entrez_ids,
#             scopes="entrezgene",
#             fields="symbol",
#             species="human"
#         )

#         # Build a mapping: input ID -> symbol (or None)
#         id_to_symbol = {}
#         for r in results:
#             q = str(r.get("query"))
#             id_to_symbol[q] = r.get("symbol") if not r.get("notfound") else None

#         # Preserve original order
#         symbols = [id_to_symbol.get(str(e), None) for e in entrez_ids]
#         comms_HGNC.append(symbols)
#     return comms_HGNC


In [13]:
COMMUNITIES_HGNC = ncbi_to_HGNC(communities_ncbi)
COMMUNITIES_HGNC_full = ncbi_to_HGNC(communities_ncbi_full)

In [14]:
print(len(COMMUNITIES_HGNC))

17


In [15]:
COMMUNITIES_HGNC = drop_nan_from_communities(COMMUNITIES_HGNC)
COMMUNITIES_HGNC_full = drop_nan_from_communities(COMMUNITIES_HGNC_full)

Community 0: dropped 0 NaN entries
Community 1: dropped 0 NaN entries
Community 2: dropped 0 NaN entries
Community 3: dropped 0 NaN entries
Community 4: dropped 0 NaN entries
Community 5: dropped 4 NaN entries
Community 6: dropped 9 NaN entries
Community 7: dropped 0 NaN entries
Community 8: dropped 3 NaN entries
Community 9: dropped 2 NaN entries
Community 10: dropped 0 NaN entries
Community 11: dropped 0 NaN entries
Community 12: dropped 1 NaN entries
Community 13: dropped 0 NaN entries
Community 14: dropped 0 NaN entries
Community 15: dropped 0 NaN entries
Community 16: dropped 0 NaN entries

Total dropped across all communities: 19
Community 0: dropped 0 NaN entries
Community 1: dropped 0 NaN entries
Community 2: dropped 4 NaN entries
Community 3: dropped 0 NaN entries
Community 4: dropped 0 NaN entries
Community 5: dropped 14 NaN entries
Community 6: dropped 11 NaN entries
Community 7: dropped 0 NaN entries
Community 8: dropped 4 NaN entries
Community 9: dropped 3 NaN entries
Comm

In [16]:
with open(f"{RESULT_FOLDER}/result_communities_HGNC_selected.pkl", "wb") as f:
    pickle.dump(COMMUNITIES_HGNC, f)
with open(f"{RESULT_FOLDER}/result_communities_HGNC.pkl", "wb") as f:
    pickle.dump(COMMUNITIES_HGNC_full, f)

In [17]:
print(len(COMMUNITIES_HGNC))
print(len(COMMUNITIES_HGNC_full))

17
424


# Categoization Prep

### GO-slim

In [18]:
DATA_DIRECTORY = "../../data"
GO_OBO = f"{DATA_DIRECTORY}/GO/go-basic.obo"            # put the file in your working dir (or give full path)
GOSLIM_OBO = f"{DATA_DIRECTORY}/GO/goslim_generic.obo"  # swap to another slim if you prefer
GOSLIM_PIR_OBO = f"{DATA_DIRECTORY}/GO/goslim_pir.obo"  # swap to another slim if you prefer
GOSLIM_YEAST_OBO = f"{DATA_DIRECTORY}/GO/goslim_yeast.obo"
GOSLIM_AGR_OBO = f"{DATA_DIRECTORY}/GO/goslim_agr.obo"

In [19]:
# GO library
go = GODag(GO_OBO)

# SLIM libraries
slim = GODag(GOSLIM_OBO)
slim_pir = GODag(GOSLIM_PIR_OBO)
slim_yeast = GODag(GOSLIM_YEAST_OBO)
slim_agr = GODag(GOSLIM_AGR_OBO)

slim_ids = set(slim.keys())
slim_pir_ids = set(slim_pir.keys())
slim_yeast_ids = set(slim_yeast.keys())
slim_agr_ids = set(slim_agr.keys())

../../data/GO/go-basic.obo: fmt(1.2) rel(2025-10-10) 42,666 Terms
../../data/GO/goslim_generic.obo: fmt(1.2) rel(go/2025-10-10/subsets/goslim_generic.owl) 205 Terms
../../data/GO/goslim_pir.obo: fmt(1.2) rel(go/2025-10-10/subsets/goslim_pir.owl) 617 Terms
../../data/GO/goslim_yeast.obo: fmt(1.2) rel(go/2025-10-10/subsets/goslim_yeast.owl) 295 Terms
../../data/GO/goslim_agr.obo: fmt(1.2) rel(go/2025-10-10/subsets/goslim_agr.owl) 94 Terms


In [20]:
GO_RE = re.compile(r"(GO:\d{7})")

def get_goid(term: str):
    if isinstance(term, str):
        m = GO_RE.search(term)
        if m:
            return m.group(1)
    raise RuntimeError("Term not found!!")

def get_go_ancestors(go_id):
    """Return a list of ancestor GO term IDs for the given GO ID using QuickGO."""
    url = f"https://www.ebi.ac.uk/QuickGO/services/ontology/go/terms/{go_id}/ancestors"
    headers = {"Accept": "application/json"}

    r = requests.get(url, headers=headers)
    r.raise_for_status()

    data = r.json()
    results = data.get("results", [])
    if not results:
        return []

    # Ancestors come back as a simple list of GO IDs (strings)
    ancestors = results[0].get("ancestors", [])
    return set(ancestors)


def get_go_ancestors_in_slim(go_id):
    ancestors = get_go_ancestors(go_id)
    return slim_ids & ancestors

In [21]:
def get_go_ancestors_at_depth(go_id, depth, include_relations=("is_a", "part_of")):
    """
    Return the set of GO term IDs that are ancestors of `go_id` and have
    absolute depth == `depth` in the GO DAG.

    Parameters
    ----------
    go_id : str
        Starting GO term (e.g., "GO:0051310").
    depth : int
        Absolute depth in the GO DAG (e.g., 3 means all ancestors at depth=3).
    include_relations : tuple[str]
        Relation types to traverse upward, e.g. ("is_a", "part_of", "regulates", ...).

    Returns
    -------
    set[str]
        Ancestor GO IDs whose term.depth == `depth`. Empty set if none.
    """
    if depth < 0:
        return set()
    if go_id not in go:
        return set()

    # One-hop function honoring relation filter
    def parent_ids(term):
        ids = set()
        if "is_a" in include_relations:
            # GOATOOLS usually puts is_a parents here (and sometimes part_of merged)
            ids.update(p.id for p in term.parents)

        rel = getattr(term, "relationship", {}) or {}
        for r in include_relations:
            # relationship entries are already GO IDs
            ids.update(rel.get(r, []))

        # ensure IDs exist in DAG
        return {pid for pid in ids if pid in go}

    result = set()
    frontier = {go_id}
    visited = {go_id}

    # BFS upwards, but pruning branches that are already above the target depth
    while frontier:
        next_frontier = set()
        for node in frontier:
            for pid in parent_ids(go[node]):
                if pid in visited:
                    continue
                visited.add(pid)
                d = go[pid].depth  # absolute depth in DAG

                if d == depth:
                    # ancestor at the exact target depth
                    result.add(pid)
                elif d > depth:
                    # still "below" target depth (further from root),
                    # its parents might reach the target depth
                    next_frontier.add(pid)
                # if d < depth: this branch has gone above the target,
                # and all further ancestors will have depth <= d, so we can skip
        frontier = next_frontier

    return result


### KEGG

In [22]:
def build_kegg_name_to_id(species="hsa"):
    """Map KEGG pathway name -> 'hsaXXXXX' (species-specific)."""
    lines = requests.get(f"https://rest.kegg.jp/list/pathway/{species}").text.strip().splitlines()
    name_to_id = {}
    for ln in lines:
        pid, raw = ln.split("\t")
        pid = pid.replace("path:", "")  # e.g. hsa03010
        # strip " - Homo sapiens (human)" suffix
        name = re.sub(r"\s*-\s*Homo sapiens.*$", "", raw).strip()
        name_to_id[name.lower()] = pid
    return name_to_id

name_to_id = build_kegg_name_to_id("hsa")

In [23]:
def get_kegg_level2(hsa_id: str) -> str | None:
    """
    Return the KEGG Level 2 category for a pathway like 'hsa03040'.
    Example: get_kegg_level2("hsa03040") -> 'Transcription'
    """
    url = f"http://rest.kegg.jp/get/{hsa_id}"
    try:
        text = requests.get(url, timeout=10).text
    except Exception:
        return None

    for line in text.splitlines():
        if line.startswith("CLASS"):
            # CLASS line looks like: CLASS       Genetic Information Processing; Transcription
            parts = [p.strip() for p in line.split(";", maxsplit=2)]
            if len(parts) >= 2:
                return [parts[1]]
            elif len(parts) == 1:
                return [parts[0].replace("CLASS", "").strip()]
    return []

### Reactome

In [24]:
def build_reactome_level_map(level=1, species="9606"):
    """
    Returns { 'R-HSA-xxxxx': ['CategoryNameAtLevel', ...], ... } for the given species.

    Parameters
    ----------
    level : int, default=1
        1-based depth in the Reactome pathway hierarchy:
          - level=1 → top-level Reactome categories (original behavior)
          - level=2 → second-level ancestors, etc.
        If a node is shallower than `level`, the deepest available ancestor
        is used as a fallback.
    species : str, default="9606"
        Taxonomy ID ("9606") or species name ("Homo sapiens").
    """
    if level < 1:
        raise ValueError("level must be >= 1 (1-based depth)")

    # ensure spaces are encoded if a name is used
    species_path = species.replace(" ", "+")
    url = f"https://reactome.org/ContentService/data/eventsHierarchy/{species_path}"
    print(url)
    r = requests.get(url, headers={"Accept": "application/json"}, timeout=300)
    r.raise_for_status()
    trees = r.json()  # list of trees, one per TopLevelPathway

    mapping = {}

    def walk(node, ancestors):
        """
        node: current node dict
        ancestors: list of ancestor nodes from root to parent of `node`
        """
        # ancestors_chain includes current node at the end
        ancestors_chain = ancestors + [node]

        st_id = node.get("stId")
        if st_id:
            # We want the ancestor at depth `level` (1-based).
            # If the path is shorter than `level`, fall back to the deepest one.
            if len(ancestors_chain) >= level:
                cat_node = ancestors_chain[level - 1]
            else:
                cat_node = ancestors_chain[-1]

            cat_name = cat_node.get("name")
            if cat_name:
                mapping.setdefault(st_id, set()).add(cat_name)

        # Recurse into children
        for child in node.get("children", []):
            walk(child, ancestors_chain)

    # Each tree is a top-level pathway
    for top in trees:
        walk(top, [])

    # sets -> sorted lists
    return {k: sorted(v) for k, v in mapping.items()}

In [25]:
# Specific for Reactome: build level map first
reactome_level1 = build_reactome_level_map(level = 1)

https://reactome.org/ContentService/data/eventsHierarchy/9606


# Run Enrichment Analysis

In [26]:
TERM_SCORE_CAP = 0.001
PERCENTAGE = 0.1

### GO

In [27]:
# GO Analysis; save terms with small size and high p-value
def go_enrichment(communities,
                  term_score_cap,
                  percentage, 
                  slim_ids = slim_yeast_ids,
                  depth = 1):
    important_terms = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value","Category"])
    i = 0
    num_nonzero_communities = 0
    
    for community in communities:
        # Gene Ontology enrichment
        enr_go = gp.enrichr(
            gene_list=community,
            gene_sets=['GO_Biological_Process_2023',
                    'GO_Molecular_Function_2023',
                    'GO_Cellular_Component_2023'],
            organism='Human',
            outdir=None # don't write to disk
        )
        go_df = enr_go.results
        

        # Filter by overlap percentage and adjusted p-value
        mask =  (go_df["Adjusted P-value"] < term_score_cap) & (go_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
        filtered = go_df[mask].copy()
        
        # Categorization from GO-Slim
        filtered["GO_ID"] = filtered["Term"].apply(get_goid)
        # filtered["Slim_IDs"] = filtered["GO_ID"].apply(get_go_ancestors_in_slim)
        filtered["Slim_IDs"] = filtered["GO_ID"].apply(lambda id: get_go_ancestors_at_depth(id, depth=depth, include_relations=("is_a", "part_of")))
        
        # Get empty count
        empty_count = (filtered["Slim_IDs"].apply(len) == 0).sum()
        
        # Get slim names    
        filtered["Category"] = filtered["Slim_IDs"].apply(lambda ids: [go[i].name for i in ids])
        
        # Sort
        filtered['Overlap (value)'] = filtered['Overlap'].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]))
        filtered = filtered.sort_values(['Overlap (value)'], ascending=False)
        
        # Add results to important terms
        if not filtered.empty:
            # print size of community
            print(f"Size of community: {len(community)}")
            
            # print number of filtered terms
            print(f"Number of filtered terms: {len(filtered)}")
            print(f"Number of unmapped terms: {empty_count}")      
            filtered.loc[:, "Community Index"] = i
            filtered.loc[:, "Community Size"] = len(community)
            important_terms = pd.concat([important_terms, filtered], ignore_index=True)
            display(HTML(filtered[["Community Index",'Term','Overlap','Adjusted P-value',"Slim_IDs","Category"]].head(10).to_html(max_cols=None)))
            num_nonzero_communities += 1

        i += 1
    print(f"{num_nonzero_communities} out of {len(communities)} communities had significant GO terms.")
    return important_terms

In [28]:
go_important_terms = go_enrichment(COMMUNITIES_HGNC,TERM_SCORE_CAP,PERCENTAGE,slim_ids,depth = 1)

Size of community: 1369
Number of filtered terms: 365
Number of unmapped terms: 10


C:\Users\celem\AppData\Local\Temp\ipykernel_31636\3881098508.py:53: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  important_terms = pd.concat([important_terms, filtered], ignore_index=True)


,Community Index,Term,Overlap,Adjusted P-value,Slim_IDs,Category
133,0,Leukocyte Aggregation (GO:0070486),6/7,2.070522e-05,{GO:0009987},[cellular process]
118,0,Wnt Signaling Pathway Involved In Midbrain Dopaminergic Neuron Differentiation (GO:1904953),7/9,7.677565e-06,{},[]
4165,0,RAGE Receptor Binding (GO:0050786),6/8,6.000369e-05,{GO:0005488},[binding]
238,0,Regulation Of Lipase Activity (GO:0060191),5/7,4.803708e-04,{},[]
235,0,Intermediate Filament Bundle Assembly (GO:0045110),5/7,4.803708e-04,{GO:0009987},[cellular process]
236,0,Positive Regulation Of Neuron Projection Arborization (GO:0150012),5/7,4.803708e-04,{GO:0065007},[biological regulation]
237,0,Positive Regulation Of Vascular Endothelial Growth Factor Signaling Pathway (GO:1900748),5/7,4.803708e-04,{GO:0065007},[biological regulation]
239,0,Regulation Of Response To Wounding (GO:1903034),5/7,4.803708e-04,{GO:0065007},[biological regulation]
4170,0,Protein Binding Involved In Heterotypic Cell-Cell Adhesion (GO:0086080),6/9,1.459765e-04,{GO:0005488},[binding]
4135,0,Frizzled Binding (GO:0005109),22/33,1.394821e-15,{GO:0005488},[binding]


Size of community: 1183
Number of filtered terms: 140
Number of unmapped terms: 12


,Community Index,Term,Overlap,Adjusted P-value,Slim_IDs,Category
45,1,snRNA Modification (GO:0040031),5/5,4.317288e-05,{GO:0009987},[cellular process]
35,1,Protein Insertion Into ER Membrane By Stop-Transfer Membrane-Anchor Sequence (GO:0045050),7/9,6.196954e-06,"{GO:0051179, GO:0009987}","[localization, cellular process]"
2778,1,2-Acylglycerol-3-Phosphate O-acyltransferase Activity (GO:0047144),5/7,5.680077e-04,{GO:0003824},[catalytic activity]
3355,1,"RNA Polymerase II, Core Complex (GO:0005665)",10/14,1.585590e-08,{GO:0032991},[protein-containing complex]
2770,1,RNA Polymerase II Activity (GO:0001055),7/10,2.479018e-05,{},[]
2767,1,U6 snRNA Binding (GO:0017070),9/13,9.630466e-07,{GO:0005488},[binding]
66,1,Positive Regulation Of rRNA Processing (GO:2000234),6/9,1.256090e-04,{GO:0065007},[biological regulation]
3363,1,Transcription Factor TFIIH Core Complex (GO:0000439),6/9,5.935198e-05,{GO:0032991},[protein-containing complex]
3353,1,Anaphase-Promoting Complex (GO:0005680),13/20,2.778874e-10,{GO:0032991},[protein-containing complex]
6,1,Regulation Of Meiotic Cell Cycle (GO:0051445),13/20,2.110875e-09,{GO:0065007},[biological regulation]


...

Size of community: 240
Number of filtered terms: 2
Number of unmapped terms: 0


,Community Index,Term,Overlap,Adjusted P-value,Slim_IDs,Category
1,12,Cell-Cell Adhesion Mediated By Cadherin (GO:0044331),5/26,0.000984,{GO:0009987},[cellular process]
222,12,Catenin Complex (GO:0016342),5/28,0.000711,{GO:0032991},[protein-containing complex]


Size of community: 96
Number of filtered terms: 9
Number of unmapped terms: 0


,Community Index,Term,Overlap,Adjusted P-value,Slim_IDs,Category
131,15,Cytosolic Large Ribosomal Subunit (GO:0022625),19/52,1.330154e-30,{GO:0032991},[protein-containing complex]
132,15,Large Ribosomal Subunit (GO:0015934),19/52,1.330154e-30,{GO:0032991},[protein-containing complex]
0,15,Cytoplasmic Translation (GO:0002181),24/93,7.612999e-34,{GO:0009987},[cellular process]
134,15,Cytosolic Small Ribosomal Subunit (GO:0022627),7/41,6.632796e-09,{GO:0032991},[protein-containing complex]
135,15,Small Ribosomal Subunit (GO:0015935),7/42,6.632796e-09,{GO:0032991},[protein-containing complex]
1,15,Peptide Biosynthetic Process (GO:0043043),26/158,1.494619e-31,{GO:0009987},[cellular process]
133,15,Ribosome (GO:0005840),9/61,1.301322e-10,{GO:0110165},[cellular anatomical structure]
2,15,Macromolecule Biosynthetic Process (GO:0009059),26/183,5.703164e-30,{GO:0009987},[cellular process]
3,15,Translation (GO:0006412),27/234,8.951119e-29,{GO:0009987},[cellular process]


10 out of 17 communities had significant GO terms.


In [29]:
go_important_terms

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,GO_ID,Slim_IDs,Overlap (value)
0,0,1369,Leukocyte Aggregation (GO:0070486),6/7,2.070522e-05,[cellular process],GO_Biological_Process_2023,6.709793e-07,0.0,0.0,82.010271,1165.737258,SEMA4D;RAC2;S100A9;CD44;JAM2;S100A8,GO:0070486,{GO:0009987},0.857143
1,0,1369,Wnt Signaling Pathway Involved In Midbrain Dop...,7/9,7.677565e-06,[],GO_Biological_Process_2023,2.209505e-07,0.0,0.0,47.871880,733.652220,FZD1;RYK;WNT5A;WNT9B;WNT2;WNT3;LRP6,GO:1904953,{},0.777778
2,0,1369,RAGE Receptor Binding (GO:0050786),6/8,6.000369e-05,[binding],GO_Molecular_Function_2023,2.527329e-06,0.0,0.0,41.002935,528.460070,S100A13;S100A12;S100A4;S100A9;S100A8;S100A7,GO:0050786,{GO:0005488},0.750000
3,0,1369,Regulation Of Lipase Activity (GO:0060191),5/7,4.803708e-04,[],GO_Biological_Process_2023,2.788125e-05,0.0,0.0,34.144062,358.087757,FURIN;PCSK6;RHOC;PCSK5;SNCA,GO:0060191,{},0.714286
4,0,1369,Intermediate Filament Bundle Assembly (GO:0045...,5/7,4.803708e-04,[cellular process],GO_Biological_Process_2023,2.788125e-05,0.0,0.0,34.144062,358.087757,KRT14;PKP2;NEFL;PKP1;NEFH,GO:0045110,{GO:0009987},0.714286
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
592,15,96,Small Ribosomal Subunit (GO:0015935),7/42,6.632796e-09,[protein-containing complex],GO_Cellular_Component_2023,1.105466e-09,0,0,44.649438,920.805314,RPS4Y2;RPS18;RPS8;RPS20;FAU;RPS11;RPS10,GO:0015935,{GO:0032991},0.166667
593,15,96,Peptide Biosynthetic Process (GO:0043043),26/158,1.494619e-31,[cellular process],GO_Biological_Process_2023,3.284876e-33,0,0,55.635498,4161.311531,RPL4;RPS4Y2;RPL3;RPL12;RPL34;RPLP1;RPL9;RPL7A;...,GO:0043043,{GO:0009987},0.164557
594,15,96,Ribosome (GO:0005840),9/61,1.301322e-10,[cellular anatomical structure],GO_Cellular_Component_2023,1.301322e-11,0,0,39.493369,989.903468,RPL7A;RPL36AL;RPS18;RPL10L;RPS11;RPL9;RPS10;GC...,GO:0005840,{GO:0110165},0.147541
595,15,96,Macromolecule Biosynthetic Process (GO:0009059),26/183,5.703164e-30,[cellular process],GO_Biological_Process_2023,1.880164e-31,0,0,46.717197,3305.184677,RPL4;RPS4Y2;RPL3;RPL12;RPL34;RPLP1;RPL9;RPL7A;...,GO:0009059,{GO:0009987},0.142077


### KEGG

In [30]:
# KEGG
def kegg_enrichment(communities,
                    term_score_cap,
                    percentage):
    important_terms = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value","Category"])
    i = 0
    num_nonzero_communities = 0
    for community in communities:
        enr_path = gp.enrichr(
            gene_list=community,
            gene_sets=['KEGG_2021_Human'],
            organism='Human',
            outdir=None
        )
        KEGG_df = enr_path.results

        # Filter by overlap percentage and adjusted p-value
        mask =  (KEGG_df["Adjusted P-value"] < term_score_cap) & (KEGG_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
        filtered = KEGG_df[mask].copy()
        
        # Categorization from KEGG Level 2
        filtered["KEGG_ID"] = filtered["Term"].str.replace(r"\s*-\s*Homo sapiens.*$", "", regex=True).str.lower().map(name_to_id)
        filtered["Category"] = filtered["KEGG_ID"].map(get_kegg_level2)
        
        # Sort
        filtered['Overlap (value)'] = filtered['Overlap'].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]))
        filtered = filtered.sort_values(['Overlap (value)'], ascending=False)
        
        # Add results to important terms
        if not filtered.empty:
            # print size of community
            print(f"Size of community: {len(community)}")   
            
            # print number of filtered terms
            print(f"Number of filtered terms: {len(filtered)}")
            filtered.loc[:, "Community Index"] = i
            filtered.loc[:, "Community Size"] = len(community)
            important_terms = pd.concat([important_terms, filtered], ignore_index=True)
            
            # show results
            display(HTML(filtered[["Community Index",'Term','Overlap','Adjusted P-value',"KEGG_ID","Category"]].head(10).to_html(max_cols=None)))
            num_nonzero_communities += 1

        i += 1
    print(f"{num_nonzero_communities} out of {len(communities)} communities had significant GO terms.")
    return important_terms

In [31]:
kegg_important_terms = kegg_enrichment(COMMUNITIES_HGNC,TERM_SCORE_CAP,PERCENTAGE)

Size of community: 1369
Number of filtered terms: 21


C:\Users\celem\AppData\Local\Temp\ipykernel_31636\1053027199.py:38: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  important_terms = pd.concat([important_terms, filtered], ignore_index=True)


,Community Index,Term,Overlap,Adjusted P-value,KEGG_ID,Category
2,0,Basal cell carcinoma,28/63,9.952302e-15,hsa05217,[Cancer: specific types]
0,0,Wnt signaling pathway,48/166,8.445888e-16,hsa04310,[Signal transduction]
3,0,Signaling pathways regulating pluripotency of stem cells,38/143,1.317007e-11,hsa04550,[Cellular community - eukaryotes]
16,0,Glycosaminoglycan biosynthesis,14/53,1.362600e-04,NaN,[]
6,0,Melanogenesis,26/101,9.845154e-08,hsa04916,[Endocrine system]
1,0,Proteoglycans in cancer,52/205,8.426211e-15,hsa05205,[Cancer: overview]
4,0,Hippo signaling pathway,39/163,1.877202e-10,hsa04390,[Signal transduction]
5,0,mTOR signaling pathway,34/154,3.733868e-08,hsa04150,[Signal transduction]
14,0,ECM-receptor interaction,19/88,1.082434e-04,hsa04512,[Signaling molecules and interaction]
7,0,Gastric cancer,31/149,6.825255e-07,hsa05226,[Cancer: specific types]


Size of community: 1183
Number of filtered terms: 17


,Community Index,Term,Overlap,Adjusted P-value,KEGG_ID,Category
2,1,DNA replication,17/36,2.095391e-10,hsa03030,[Replication and repair]
0,1,N-Glycan biosynthesis,22/50,2.577027e-12,hsa00510,[Glycan biosynthesis and metabolism]
9,1,Mismatch repair,10/23,5.116401e-06,hsa03430,[Replication and repair]
1,1,Nucleotide excision repair,20/47,4.454300e-11,hsa03420,[Replication and repair]
3,1,Basal transcription factors,18/45,1.160074e-09,hsa03022,[Transcription]
6,1,Various types of N-glycan biosynthesis,15/39,5.852777e-08,hsa00513,[Glycan biosynthesis and metabolism]
10,1,RNA polymerase,11/31,1.367447e-05,hsa03020,[Transcription]
16,1,Base excision repair,9/33,9.635387e-04,hsa03410,[Replication and repair]
13,1,Amino sugar and nucleotide sugar metabolism,12/48,2.131004e-04,hsa00520,[Glycan biosynthesis and metabolism]
8,1,mRNA surveillance pathway,23/98,1.883846e-07,hsa03015,[Translation]


...

Size of community: 220
Number of filtered terms: 1


,Community Index,Term,Overlap,Adjusted P-value,KEGG_ID,Category
0,11,Olfactory transduction,213/440,0.0,hsa04740,[Sensory system]


Size of community: 96
Number of filtered terms: 2


,Community Index,Term,Overlap,Adjusted P-value,KEGG_ID,Category
0,15,Ribosome,30/158,1.244493e-39,hsa03010,[Translation]
1,15,Coronavirus disease,29/232,5.768266e-33,NaN,[]


7 out of 17 communities had significant GO terms.


In [32]:
kegg_important_terms

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,KEGG_ID,Overlap (value)
0,0,1369,Basal cell carcinoma,28/63,9.952302e-15,[Cancer: specific types],KEGG_2021_Human,1.143943e-16,0.0,0.0,11.093811,407.219181,WNT2B;HHIP;WNT8A;FZD10;WNT6;SHH;DVL1;DVL2;DVL3...,hsa05217,0.444444
1,0,1369,Wnt signaling pathway,48/166,8.445888e-16,[Signal transduction],KEGG_2021_Human,3.235973e-18,0.0,0.0,5.700766,229.582400,WNT2B;CTNND2;WNT8A;FZD10;NLK;NKD1;LRP6;WNT6;CC...,hsa04310,0.289157
2,0,1369,Signaling pathways regulating pluripotency of ...,38/143,1.317007e-11,[Cellular community - eukaryotes],KEGG_2021_Human,2.018401e-13,0.0,0.0,5.037301,147.246860,WNT2B;WNT8A;FZD10;IGF1R;WNT6;ACVR1C;DVL1;DVL2;...,hsa04550,0.265734
3,0,1369,Glycosaminoglycan biosynthesis,14/53,1.362600e-04,[],KEGG_2021_Human,8.875173e-06,0.0,0.0,4.925499,57.294650,HS3ST3B1;CHST7;HS3ST3A1;XYLT1;HS6ST1;HS6ST2;EX...,NaN,0.264151
4,0,1369,Melanogenesis,26/101,9.845154e-08,[Endocrine system],KEGG_2021_Human,2.640463e-09,0.0,0.0,4.789834,94.610288,WNT2B;WNT8A;FZD10;WNT6;DVL1;DVL2;DVL3;WNT2;WNT...,hsa04916,0.257426
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,7,456,Necroptosis,17/159,9.527049e-07,[Cell growth and death],KEGG_2021_Human,1.424605e-07,0.0,0.0,5.291058,83.409307,H2AZ2;H2AC8;H2AC6;H2AC7;H2AC4;H2AC19;H2AC1;H2A...,hsa04217,0.106918
60,7,456,MicroRNAs in cancer,32/310,1.313418e-11,[Cancer: overview],KEGG_2021_Human,1.104744e-12,0.0,0.0,5.230352,143.998938,MIR29B1;MIR29B2;MIR17;MIR34C;MIR34B;MIR18A;MIR...,hsa05206,0.103226
61,11,220,Olfactory transduction,213/440,0.000000e+00,[Sensory system],KEGG_2021_Human,0.000000e+00,0.0,0.0,2621.012587,inf,OR7G1;OR8I2;OR9K2;OR11H2;OR11H1;OR2M7;OR11H4;O...,hsa04740,0.484091
62,15,96,Ribosome,30/158,1.244493e-39,[Translation],KEGG_2021_Human,2.074155e-40,0.0,0.0,70.227273,6416.936267,RPL4;RPS4Y2;RPL3;RPL12;RPL34;RPLP1;RPL10L;RPL3...,hsa03010,0.189873


### Reactome

In [33]:
# Reactome enrichment
def reactome_enrichment(communities,
                        term_score_cap,
                        percentage):
    important_terms = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value","Category"])
    i = 0
    num_nonzero_communities = 0
    for community in communities:
        enr_path = gp.enrichr(
            gene_list=community,
            gene_sets=['Reactome_2022'],
            organism='Human',
            outdir=None
        )
        Reactome_df = enr_path.results

        # Filter by overlap percentage and adjusted p-value
        mask =  (Reactome_df["Adjusted P-value"] < term_score_cap) & (Reactome_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
        filtered = Reactome_df[mask].copy()
        
        # Categorization from Reactome Level 1
        filtered["Category"] = filtered["Term"].str.extract(r"(R-[A-Z]+-\d+)", expand=False).map(reactome_level1)
        
        # Sort
        filtered['Overlap (value)'] = filtered['Overlap'].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]))
        filtered = filtered.sort_values(['Overlap (value)'], ascending=False)
        
        # Add results to important terms
        if not filtered.empty:
            print(f"Size of community: {len(community)}")
            print(f"Number of filtered terms: {len(filtered)}")
            filtered.loc[:, "Community Index"] = i
            filtered.loc[:, "Community Size"] = len(community)
            important_terms = pd.concat([important_terms, filtered], ignore_index=True)
            display(HTML(filtered[["Community Index",'Term','Overlap','Adjusted P-value',"Category"]].head(30).to_html(max_cols=None)))
            num_nonzero_communities += 1
        i += 1
    print(f"{num_nonzero_communities} out of {len(communities)} communities had significant GO terms.")
    return important_terms

In [34]:
reactome_important_terms = reactome_enrichment(COMMUNITIES_HGNC,TERM_SCORE_CAP,PERCENTAGE)

Size of community: 1369
Number of filtered terms: 62


C:\Users\celem\AppData\Local\Temp\ipykernel_31636\3965740400.py:34: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  important_terms = pd.concat([important_terms, filtered], ignore_index=True)


,Community Index,Term,Overlap,Adjusted P-value,Category
18,0,Sema4D Induced Cell Migration And Growth-Cone Collapse R-HSA-416572,11/20,8.565857e-07,[Developmental Biology]
45,0,Dissolution Of Fibrin Clot R-HSA-75205,7/13,2.078938e-04,[Hemostasis]
44,0,"Defective EXT1 Causes Exostoses 1, TRPS2 And CHDS R-HSA-3656253",7/13,2.078938e-04,[Disease]
14,0,WNT Ligand Biogenesis And Trafficking R-HSA-3238698,14/26,1.589335e-08,[Signal Transduction]
17,0,Sema4D In Semaphorin Signaling R-HSA-400685,12/24,8.118583e-07,[Developmental Biology]
53,0,TRAF3-dependent IRF Activation Pathway R-HSA-918233,7/15,5.555088e-04,[Immune System]
5,0,Interferon Alpha/Beta Signaling R-HSA-909733,29/72,1.580924e-13,[Immune System]
57,0,Regulation Of FZD By Ubiquitination R-HSA-4641263,8/20,5.555088e-04,[Signal Transduction]
58,0,Syndecan Interactions R-HSA-3000170,8/20,5.555088e-04,[Extracellular matrix organization]
54,0,Defective B3GALT6 Causes EDSP2 And SEMDJL1 R-HSA-4420332,8/20,5.555088e-04,[Disease]


Size of community: 1183
Number of filtered terms: 169


,Community Index,Term,Overlap,Adjusted P-value,Category
97,1,Ubiquinol Biosynthesis R-HSA-2142789,6/8,9.756795e-06,[Metabolism]
23,1,RNA Pol II CTD Phosphorylation And Interaction With CE R-HSA-77075,19/27,2.126503e-16,[Gene expression (Transcription)]
21,1,mRNA Capping R-HSA-72086,20/29,5.753668e-17,[Metabolism of RNA]
45,1,Conversion From APC/C:Cdc20 To APC/C:Cdh1 In Late Anaphase R-HSA-176407,13/20,1.039754e-10,[Cell Cycle]
22,1,Formation Of Early Elongation Complex R-HSA-113418,21/33,9.556107e-17,[Gene expression (Transcription)]
51,1,Signaling By FGFR2 IIIa TM R-HSA-8851708,12/19,1.022712e-09,[Disease]
147,1,DNA Replication Initiation R-HSA-68952,5/8,2.091940e-04,"[Cell Cycle, DNA Replication]"
49,1,Inactivation Of APC/C Via Direct Inhibition Of APC/C Complex R-HSA-141430,13/21,2.374811e-10,[Cell Cycle]
43,1,Abortive Elongation Of HIV-1 Transcript In Absence Of Tat R-HSA-167242,14/23,5.985809e-11,[Disease]
52,1,Aberrant Regulation Of Mitotic Exit In Cancer Due To RB1 Defects R-HSA-9687136,12/20,2.373211e-09,[Disease]


...

Size of community: 240
Number of filtered terms: 5


,Community Index,Term,Overlap,Adjusted P-value,Category
3,12,Adherens Junctions Interactions R-HSA-418990,7/29,3.170690e-07,[Cell-Cell communication]
0,12,Cell-cell Junction Organization R-HSA-421270,13/61,9.438996e-12,[Cell-Cell communication]
4,12,Tight Junction Interactions R-HSA-420029,5/30,1.650317e-04,[Cell-Cell communication]
1,12,Cell Junction Organization R-HSA-446728,13/86,4.800581e-10,[Cell-Cell communication]
2,12,Cell-Cell Communication R-HSA-1500931,13/120,2.246934e-08,[Cell-Cell communication]


Size of community: 96
Number of filtered terms: 28


,Community Index,Term,Overlap,Adjusted P-value,Category
1,15,Peptide Chain Elongation R-HSA-156902,28/86,1.265272e-43,[Metabolism of proteins]
2,15,Selenocysteine Synthesis R-HSA-2408557,28/90,1.906778e-43,[Metabolism]
3,15,Viral mRNA Translation R-HSA-192823,28/90,1.906778e-43,[Disease]
4,15,Eukaryotic Translation Elongation R-HSA-156842,28/90,1.906778e-43,[Metabolism of proteins]
5,15,Eukaryotic Translation Termination R-HSA-72764,28/90,1.906778e-43,[Metabolism of proteins]
0,15,Response Of EIF2AK4 (GCN2) To Amino Acid Deficiency R-HSA-9633012,30/98,1.192075e-45,[Cellular responses to stimuli]
6,15,Nonsense Mediated Decay (NMD) Independent Of Exon Junction Complex (EJC) R-HSA-975956,28/92,3.371507e-43,[Metabolism of RNA]
9,15,Formation Of A Pool Of Free 40S Subunits R-HSA-72689,28/98,1.857874e-42,[Metabolism of proteins]
7,15,L13a-mediated Translational Silencing Of Ceruloplasmin Expression R-HSA-156827,29/108,4.996477e-43,[Metabolism of proteins]
8,15,GTP Hydrolysis And Joining Of 60S Ribosomal Subunit R-HSA-72706,29/109,6.031769e-43,[Metabolism of proteins]


12 out of 17 communities had significant GO terms.


In [35]:
reactome_important_terms

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,Overlap (value)
0,0,1369,Sema4D Induced Cell Migration And Growth-Cone ...,11/20,8.565857e-07,[Developmental Biology],Reactome_2022,1.403028e-08,0.0,0.0,16.760105,303.057015,ARHGEF11;ARHGEF12;SEMA4D;LIMK2;MYH14;MYH11;PLX...,0.550000
1,0,1369,Dissolution Of Fibrin Clot R-HSA-75205,7/13,2.078938e-04,[Hemostasis],Reactome_2022,8.244064e-06,0.0,0.0,15.953867,186.756238,ANXA2;SERPINE2;PLAU;PLAUR;PLAT;SERPINB8;SERPINB6,0.538462
2,0,1369,"Defective EXT1 Causes Exostoses 1, TRPS2 And C...",7/13,2.078938e-04,[Disease],Reactome_2022,8.244064e-06,0.0,0.0,15.953867,186.756238,EXT1;SDC4;GPC3;SDC1;GPC4;AGRN;GPC6,0.538462
3,0,1369,WNT Ligand Biogenesis And Trafficking R-HSA-32...,14/26,1.589335e-08,[Signal Transduction],Reactome_2022,2.055175e-10,0.0,0.0,16.031119,357.581969,WNT10A;WNT2B;WNT7B;WNT5A;WNT8A;WNT9B;WNT7A;WNT...,0.538462
4,0,1369,Sema4D In Semaphorin Signaling R-HSA-400685,12/24,8.118583e-07,[Developmental Biology],Reactome_2022,1.259780e-08,0.0,0.0,13.720707,249.576149,ARHGEF11;ARHGEF12;SEMA4D;RRAS;LIMK2;MYH14;MYH1...,0.500000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
601,15,96,Translation Initiation Complex Formation R-HSA...,8/57,5.031617e-10,[Metabolism of proteins],Reactome_2022,2.851250e-10,0.0,0.0,36.836735,809.601201,RPS4Y2;RPS8;RPS20;FAU;RPS11;EIF2S2;RPS10;RPS12,0.140351
602,15,96,Ribosomal Scanning And Start Codon Recognition...,8/57,5.031617e-10,[Metabolism of proteins],Reactome_2022,2.851250e-10,0.0,0.0,36.836735,809.601201,RPS4Y2;RPS8;RPS20;FAU;RPS11;EIF2S2;RPS10;RPS12,0.140351
603,15,96,mRNA Activation Upon Binding Of Cap-Binding Co...,8/58,5.647752e-10,[Metabolism of proteins],Reactome_2022,3.294522e-10,0.0,0.0,36.098182,788.152899,RPS4Y2;RPS8;RPS20;FAU;RPS11;EIF2S2;RPS10;RPS12,0.137931
604,15,96,Signaling By ROBO Receptors R-HSA-376176,28/209,1.031693e-32,[Developmental Biology],Reactome_2022,3.782875e-33,0.0,0.0,44.868703,3349.665133,RPL4;RPS4Y2;RPL3;RPL12;RPL34;RPLP1;RPL10L;RPL3...,0.133971


### Disease Data Sets

In [36]:
# disease_term_score_cap = 0.001
# disease_percentage = 0.1
# important_diseases = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value"])

In [37]:
# # Disease-gene enrichment libraries
# disease_sets = [
#     'DisGeNET_2020', # curated gene–disease associations
#     'GWAS_Catalog_2023', # genome-wide association hits
#     'OMIM_Disease', # Mendelian disorders
#     'Jensen_DISEASES' # text-mined associations
# ]

# # # Disease-gene enrichment Analysis; save terms with small size and high p-value
# i = 0
# for community in communities_HGNC:
#     # Gene Ontology enrichment
#     enr_disease = gp.enrichr(
#         gene_list=community,
#         gene_sets=disease_sets,
#         organism='Human',
#         outdir=None # don't write to disk
#     )
#     enr_disease_df = enr_disease.results.sort_values('Adjusted P-value')
#     print(f"Size of community: {len(community)}")

#     mask =  (enr_disease_df["Adjusted P-value"] < disease_term_score_cap) & (enr_disease_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > disease_percentage))
        
#     filtered = enr_disease_df[mask].copy()
#     if not filtered.empty:
#         filtered.loc[:, "Community Index"] = i
#         filtered.loc[:, "Community Size"] = len(community)
#         important_diseases = pd.concat([important_diseases, filtered], ignore_index=True)

#     display(HTML(filtered[['Term','Overlap','Adjusted P-value']].head(10).to_html(max_cols=None)))
#     i += 1

# Important Terms df

In [38]:
important_terms = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value","Category"])
c = [go_important_terms,kegg_important_terms,reactome_important_terms]
important_terms = pd.concat(c, ignore_index=True)
important_terms = important_terms.sort_values(by="Community Index")
important_terms

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,GO_ID,Slim_IDs,Overlap (value),KEGG_ID
0,0,1369,Leukocyte Aggregation (GO:0070486),6/7,2.070522e-05,[cellular process],GO_Biological_Process_2023,6.709793e-07,0.0,0.0,82.010271,1165.737258,SEMA4D;RAC2;S100A9;CD44;JAM2;S100A8,GO:0070486,{GO:0009987},0.857143,NaN
305,0,1369,Regulation Of Intracellular Signal Transductio...,48/297,1.080583e-06,[biological regulation],GO_Biological_Process_2023,2.247403e-08,0.0,0.0,2.682451,47.240395,TRIO;PHLPP1;DOCK8;ITSN1;ARHGAP1;ARHGAP18;ARHGE...,GO:1902531,{GO:0065007},0.161616,NaN
304,0,1369,Defense Response To Bacterium (GO:0042742),33/204,8.624171e-05,"[response to stimulus, biological process invo...",GO_Biological_Process_2023,3.357900e-06,0.0,0.0,2.666509,33.609200,ANKRD17;COLEC12;RAB1A;DEFB1;LYST;SPN;RNF213;RP...,GO:0042742,"{GO:0050896, GO:0044419}",0.161765,NaN
303,0,1369,Positive Regulation Of Macromolecule Metabolic...,59/364,3.955200e-08,[biological regulation],GO_Biological_Process_2023,4.973891e-10,0.0,0.0,2.706129,57.969753,MYOM1;PHB1;GSK3A;TFRC;CITED2;CELF1;HFE;PDCD5;E...,GO:0010604,{GO:0065007},0.162088,NaN
302,0,1369,Protein Kinase Binding (GO:0019901),83/511,4.573627e-11,[binding],GO_Molecular_Function_2023,1.242834e-13,0.0,0.0,2.744962,81.569871,ERRFI1;PTPRR;GSK3A;TFRC;FAF1;ILK;WDR45;LYST;EL...,GO:0019901,{GO:0005488},0.162427,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1242,15,96,Eukaryotic Translation Elongation R-HSA-156842,28/90,1.906778e-43,[Metabolism of proteins],Reactome_2022,1.906778e-44,0.0,0.0,131.777989,13265.869916,RPL4;RPL3;RPS4Y2;RPL34;RPLP1;RPL12;RPL10L;RPL3...,NaN,NaN,0.311111,NaN
1243,15,96,Eukaryotic Translation Termination R-HSA-72764,28/90,1.906778e-43,[Metabolism of proteins],Reactome_2022,1.906778e-44,0.0,0.0,131.777989,13265.869916,RPL4;RPL3;RPS4Y2;RPL34;RPLP1;RPL12;RPL10L;RPL3...,NaN,NaN,0.311111,NaN
1244,15,96,Response Of EIF2AK4 (GCN2) To Amino Acid Defic...,30/98,1.192075e-45,[Cellular responses to stimuli],Reactome_2022,1.986792e-47,0.0,0.0,132.593583,14258.448024,RPL4;RPS4Y2;RPL3;RPL12;RPL34;RPLP1;RPL10L;RPL3...,NaN,NaN,0.306122,NaN
1246,15,96,Formation Of A Pool Of Free 40S Subunits R-HSA...,28/98,1.857874e-42,[Metabolism of proteins],Reactome_2022,3.096456e-43,0.0,0.0,116.670588,11419.822259,RPL4;RPS4Y2;RPL3;RPL12;RPL34;RPLP1;RPL10L;RPL3...,NaN,NaN,0.285714,NaN


In [39]:
important_terms.to_csv(f"../output/{DISEASE}/important_terms.csv", index=False)

# Robustness Analysis

In [ ]:
# def run_enrichment_func(community,term_score_cap,percentage):
#     # GO df
#     enr_go = gp.enrichr(
#         gene_list=community,
#         gene_sets=['GO_Biological_Process_2023',
#                 'GO_Molecular_Function_2023',
#                 'GO_Cellular_Component_2023'],
#         organism='Human',
#         outdir=None # don't write to disk
#     )
#     GO_df = enr_go.results
#     mask =  (GO_df["Adjusted P-value"] < term_score_cap) & (GO_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
#     GO_df = GO_df[mask].copy()   
    
#     # KEGG df
#     enr_kegg = gp.enrichr(
#         gene_list=community,
#         gene_sets=['KEGG_2021_Human'],
#         organism='Human',
#         outdir=None
#     )
#     KEGG_df = enr_kegg.results
#     mask =  (KEGG_df["Adjusted P-value"] < term_score_cap) & (KEGG_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
#     KEGG_df = KEGG_df[mask].copy() 
       
#     # Reactome df
#     enr_reactome = gp.enrichr(
#         gene_list=community,
#         gene_sets=['Reactome_2022'],
#         organism='Human',
#         outdir=None
#     )
#     Reactome_df = enr_reactome.results  
#     mask =  (Reactome_df["Adjusted P-value"] < term_score_cap) & (Reactome_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
#     Reactome_df = Reactome_df[mask].copy()
    
    
#     all_df = [GO_df,KEGG_df,Reactome_df]
#     # build result df by concatenating
#     result = pd.concat(all_df, ignore_index=True)
#     return result

In [ ]:
# from json import JSONDecodeError

# # ---------------- 1) Safe wrapper that calls YOUR enrichr function ----------------
# _ENR_CACHE = {}  # key: tuple(sorted(genes)) -> DataFrame (copy)

# def run_enrichment_safe(run_enrichment_func, community, retries=5, base_sleep=0.8):
#     """
#     Calls user's run_enrichment_func(community) with retries + memoization.
#     Returns a DataFrame (possibly empty). Never raises JSONDecodeError outward.
#     """
#     # Ensure we always pass a list of gene symbols (never a bare string)
#     genes = np.atleast_1d(np.array(community, dtype=object)).tolist()
#     if len(genes) == 0:
#         return pd.DataFrame()

#     key = tuple(sorted(genes))
#     if key in _ENR_CACHE:
#         return _ENR_CACHE[key].copy()

#     for a in range(retries):
#         try:
#             df = run_enrichment_func(genes,TERM_SCORE_CAP,PERCENTAGE)
#             if df is None:
#                 # treat as transient failure to trigger retry
#                 raise RuntimeError("run_enrichment_func returned None")
#             _ENR_CACHE[key] = df.copy()
#             return df
#         except (JSONDecodeError, OSError, RuntimeError, ValueError) as e:
#             # Transient errors from HTTP/JSON/file handling inside gseapy
#             if a == retries - 1:
#                 # Give up: return empty so pipeline continues
#                 return pd.DataFrame()
#             time.sleep(base_sleep * (2 ** a) + np.random.rand() * 0.3)

#     return pd.DataFrame()

# # ---------------- 2) Minimal bootstrap to record robust terms ----------------
# def get_robust_terms(communities_HGNC, run_enrichment_func,
#                      R=50, leaveout=0.10, recurrence_cutoff=0.70, seed=42):
#     """
#     Uses YOUR run_enrichment_func(community)->DataFrame (already filtered to significant terms).
#     Returns DataFrame with columns: community_id, term, recurrence (and Gene_set if available).
#     """
#     rng = np.random.default_rng(seed)
#     rows = []

#     for cid, community in enumerate(communities_HGNC):
#         n = len(community)
#         if n == 0:
#             continue
#         drop_k = max(1, int(np.floor(leaveout * n)))
#         counts = Counter()

#         for _ in range(R):
#             # Jackknife subset (ensure not empty)
#             keep = np.ones(n, dtype=bool)
#             keep[rng.choice(n, size=min(drop_k, n), replace=False)] = False
#             sub = np.atleast_1d(np.array(community, dtype=object)[keep]).tolist()
#             if len(sub) == 0:
#                 continue

#             df = run_enrichment_safe(run_enrichment_func, sub)
#             if df is None or df.empty:
#                 continue

#             # Your function already returns significant terms; just count them.
#             # If it includes multiple libraries, preserve Gene_set to disambiguate names.
#             if 'Term' not in df.columns:
#                 continue  # be defensive

#             if 'Gene_set' in df.columns:
#                 terms = (df[['Term', 'Gene_set']]
#                          .dropna()
#                          .drop_duplicates()
#                          .apply(lambda r: f"{r['Term']}|{r['Gene_set']}", axis=1)
#                          .tolist())
#             else:
#                 terms = df['Term'].dropna().drop_duplicates().tolist()

#             counts.update(terms)

#             # tiny pause helps with API rate limits if your func calls Enrichr internally
#             time.sleep(0.03)

#         # Keep only robust terms
#         for t, c in counts.items():
#             freq = c / max(R, 1)
#             if freq >= recurrence_cutoff:
#                 if '|' in t:
#                     term, gene_set = t.split('|', 1)
#                     rows.append({'Community Index': cid, 'Term': term, 'recurrence': freq, 'Gene_set': gene_set})
#                 else:
#                     rows.append({'Community Index': cid, 'Term': t, 'recurrence': freq})

#     return (pd.DataFrame(rows)
#               .sort_values(['Community Index', 'recurrence'], ascending=[True, False])
#               .reset_index(drop=True))

In [ ]:
# twr3 = get_robust_terms([COMMUNITIES_HGNC[1]], run_enrichment_func,
#                                 R=25, leaveout=0.1, recurrence_cutoff=0)

In [ ]:
# twr3

In [ ]:
# terms_with_recurrence = get_robust_terms(COMMUNITIES_HGNC, run_enrichment_func,
#                                 R=10, leaveout=0.1, recurrence_cutoff=0)

In [ ]:
# terms_with_recurrence

In [ ]:
# # rename important terms to match terms_with_recurrence
# important_terms = important_terms.rename(columns={'index': 'community_id'})
# important_terms = important_terms.rename(columns={'Term': 'term'})

In [ ]:
# terms_with_rec_merged = important_terms.merge(
#     terms_with_recurrence[['community_id', 'term', 'Gene_set', 'recurrence']],
#     on=['community_id', 'term', 'Gene_set'],
#     how='left'
# )

# terms_with_rec_merged['recurrence'] = terms_with_rec_merged['recurrence'].fillna(0.0)

# terms_with_rec_merged = terms_with_rec_merged.sort_values(
#     ['community_id', 'recurrence'],
#     ascending=[True, False]
# ).reset_index(drop=True)

In [ ]:
# terms_with_rec_merged

In [ ]:
# community_summary = (
#     terms_with_rec_merged
#     .groupby("community_id")["recurrence"]
#     .agg(mean_recurrence="mean", term_count="count")
#     .reset_index()
# )

# print(community_summary)

In [ ]:
# display(HTML(terms_with_recurrence.to_html(max_cols=None)))

# Checks!

In [ ]:
DGIDB_genes_ncbi = list(DGIDB_gene_to_index.keys())

In [ ]:
def DGIDB_count(c):
    return len(set(c) & set(DGIDB_genes_ncbi))